# Weak Report Labels: A Path to a Better Score (Pending Validation)

![Overview](https://gastigado.cnies.org/d/public/ds.png)

Of the 4,407 training studies, only **58 carry gold labels**; the rest are images + multilingual reports.
Everyone builds weak labels from the reports to train a vision model. This notebook lays out a
**pending-validation** path toward a better score:

> **Multi-model report extraction → disagreement arbitration → conf into training → end-to-end validation**

---

> **Credits & attribution:** This notebook builds entirely on
> [Pilkwang Kim's rsna-knee-baseline-v1](https://www.kaggle.com/code/pilkwang/rsna-knee-baseline-v1)
> and its LLM-label dataset [rsna-knee-llm-labels](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels):
> - The extractor's **prompt, schema, verdict/severity definitions** come verbatim from Pilkwang's `api_labeler.py`
> - The extractor in Section 3 is **an adaptation of Pilkwang's `api_labeler.py`** (only the LLM endpoint changed; the logic is unchanged)
> - `report_labels_v2.csv` in the 58-study comparison is the Claude LLM labels Pilkwang released
> - Our addition is re-running four LLMs and proposing the disagreement-ensemble / conf path, all within Pilkwang's framework

## 1. Hypothesis: will better report extraction improve the score?

**Why it might help:**
- Training relies entirely on weak labels. Weak labels are **noisy targets** — less noise means cleaner signal.
- On the 58 gold studies, report-vs-label agreement is only ~82.5% (labels are image-derived).
  The closer an extractor gets to that ceiling, the more trustworthy the weak labels.

**But "better" must be validated end-to-end:**
Extraction AUC → training → final macro AUC, with **attenuation in between**.
A 0.005 extraction gap on 58 studies is noise (nekkon: σ=0.0125). To confirm a score gain,
you must **swap weak labels → train the same vision model → compare final macro AUC**,
not just look at extraction scores.

## 2. 58-study comparison: which extractor is stronger

Five extractors are aligned on the same gold studies. Each model's scores come from local caches (the audit CSV).
`report_labels_v2.csv` is Pilkwang's Claude LLM labels; `gold58_audit.csv` is our re-run of four LLMs.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

# Data mount paths (adjust to your actual mount layout)
COMP = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")            # competition data
GOLD_AUDIT = Path("/kaggle/input/datasets/cjlcjlcjl/gold58-audit")      # audit CSV (5 models)
PILKWANG = Path("/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels")  # Pilkwang LLM labels

train = pd.read_csv(COMP / "train.csv")
audit = pd.read_csv(GOLD_AUDIT / "gold58_audit.csv")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
           "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis",
           "Baker's", "Contusion", "Fracture"]
gold = train.dropna(subset=TARGETS).reset_index(drop=True)
print(f"gold studies: {len(gold)}")

# Pilkwang CSV coverage + the missing study
csv = pd.read_csv(PILKWANG / "report_labels_v2.csv")
missing_pilkwang = set(gold.StudyInstanceUID) - set(csv.StudyInstanceUID)
print(f"Pilkwang CSV missing on gold: {len(missing_pilkwang)} study(ies)")

def macro_auc(Y, S):
    aucs = []
    for j in range(len(TARGETS)):
        if Y[:, j].sum() in (0, len(Y)):
            continue
        try:
            aucs.append(roc_auc_score(Y[:, j], S[:, j]))
        except ValueError:
            pass
    return float(np.mean(aucs)) if aucs else float("nan")

Y = gold[TARGETS].values.astype(int)
print("\n=== coverage + macro AUC (each on its full subset) ===")
print(f"  {'model':20s} {'cov':>6s} {'macroAUC':>9s}")
for m, disp in [("glm","GLM"),("ds_pro","DS pro"),
                ("ds_flash","DS flash (thinking)"),("ds_flash_nothink","DS flash (no-think)"),
                ("minimax","MiniMax")]:
    cols = [f"{m}_{t}" for t in TARGETS]
    S = audit[cols].values
    keep = ~np.isnan(S).any(axis=1)
    a = macro_auc(Y[keep], S[keep])
    print(f"  {disp:20s} {keep.sum():>4d}/58 {a:>9.4f}")

csvv = csv.set_index("StudyInstanceUID").reindex(gold.StudyInstanceUID)[TARGETS].values.astype(float)
keep = ~np.isnan(csvv).any(axis=1)
a = macro_auc(Y[keep], csvv[keep])
print(f"  {'Pilkwang CSV (Claude)':20s} {keep.sum():>4d}/58 {a:>9.4f}")

## 3. LLM extractor (adapted from Pilkwang's `api_labeler.py`)

The complete extractor follows. The **prompt, 12-key schema, verdict/severity definitions, and
batch reading logic all come from Pilkwang's `api_labeler.py`**. We made two adaptations:
1. Model switched to **DeepSeek v4-flash in non-thinking mode**
2. Dropped `output_config` (Anthropic-compatible endpoints do not support JSON schema;
   enforced via prompt + tolerant parsing instead)

**This cell requires a local API key to run** (no key in the Kaggle environment; skip it there).

In [ ]:
"""Adapted from pilkwang/rsna-knee-llm-labels/api_labeler.py (LLM extractor).

The original used Claude opus-5 (Anthropic SDK). Here we use DeepSeek v4-flash in
non-thinking mode; the rest (prompt/schema/verdict→score mapping/batch) is unchanged.
"""
import json
import os
import threading
from pathlib import Path

# ---- snake_case keys for the 12 findings (order matches TARGETS) ----
KEYS = ["acl", "mcl", "medial_meniscus", "lateral_meniscus",
        "medial_oa", "lateral_oa", "pf_oa", "effusion",
        "synovitis", "bakers", "contusion", "fracture"]

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
           "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis",
           "Baker's", "Contusion", "Fracture"]

# ---- SYSTEM prompt (verbatim from Pilkwang's api_labeler.py) ----
FINDING_SPEC = """For each finding below, decide what THE REPORT asserts:
- ACL: tear or rupture of the anterior cruciate ligament
- MCL: tear or sprain of the medial collateral ligament
- medial_meniscus: tear of the medial meniscus
- lateral_meniscus: tear of the lateral meniscus
- medial_oa / lateral_oa / pf_oa: osteoarthritis (cartilage loss) in that compartment
- effusion: joint effusion
- synovitis: inflammation or thickening of the synovial lining
- bakers: Baker (popliteal) cyst
- contusion: bone contusion / bone marrow oedema from impact
- fracture: acute fracture or cortical break"""

SYSTEM = """You are an expert musculoskeletal radiologist reading a knee MRI report.
Reports may be written in any language, including Greek, Bulgarian, Turkish, Croatian,
Dutch, German, Spanish, French or English. Read each one in its original language.

For each of the twelve findings below, state what THE REPORT IN FRONT OF YOU says.
{spec}
verdict: YES the report asserts it; NO the report denies it; UNK it is not mentioned.
severity (0 unless YES): 1 minimal/mild, 2 moderate, 3 marked/severe/full-thickness.
Respond with JSON only: one object with all twelve keys, each {{"verdict","severity"}}.
""".format(spec=FINDING_SPEC)

# ---- tolerant parsing (model output keys are not consistently cased / structured) ----
import re as _re

def parse_obj(text: str) -> dict:
    """Parse a 12-key object from model output. Handles mixed-case keys / ```json /
    array-wrapped / collapsed keys.

    Anthropic-compatible endpoints do not enforce a JSON schema, so the model may output:
      - mixed-case keys (ACL / medial_meniscus)
      - the 12 keys wrapped in an array [{...}]
      - keys collapsed to a single letter (e.g. all "l")
    Falls back layer by layer, finally scraping "KEY: verdict" pairs with regex.
    """
    t = text.strip()
    if t.startswith("```"):
        m = _re.match(r"```(?:json)?\s*(.*?)\s*```", t, _re.S)
        if m:
            t = m.group(1).strip()
    try:
        obj = json.loads(t)
        if isinstance(obj, list) and len(obj) == 1 and isinstance(obj[0], dict):
            obj = obj[0]
        if isinstance(obj, dict):
            alias = {k.lower(): k for k in KEYS}
            norm = {}
            for k, v in obj.items():
                key = k.lower().replace("'", "").replace(" ", "_").replace("-", "_")
                if key in alias:
                    norm[alias[key]] = v
            if all(k in norm for k in KEYS):
                return norm
            vals = list(obj.values())
            if len(vals) == 12 and all(isinstance(v, dict) and "verdict" in v for v in vals):
                return {k: {"verdict": v["verdict"], "severity": v.get("severity", 0)}
                        for k, v in zip(KEYS, vals)}
    except json.JSONDecodeError:
        pass
    # whole-text regex fallback
    out = {}
    for k in KEYS:
        pat = rf'"?{_re.escape(k)}"?(?!\w)\s*:\s*\{{[^}}]*?"verdict"\s*:\s*"(YES|NO|UNK)"[^}}]*?"severity"\s*:\s*([0-3])[^}}]*\}}'
        m = _re.search(pat, text, _re.S | _re.I)
        if not m:
            pat2 = rf'"?{_re.escape(k)}"?(?!\w)\s*:\s*\{{[^}}]*?"severity"\s*:\s*([0-3])[^}}]*?"verdict"\s*:\s*"(YES|NO|UNK)"[^}}]*\}}'
            m = _re.search(pat2, text, _re.S | _re.I)
        if m:
            if m.group(1) in ("YES", "NO", "UNK"):
                out[k] = {"verdict": m.group(1), "severity": int(m.group(2))}
            else:
                out[k] = {"verdict": m.group(2), "severity": int(m.group(1))}
    if all(k in out for k in KEYS):
        return out
    raise ValueError(f"could not parse a 12-key object from output:\\n{text[:400]}")


# ---- verdict/severity -> score (5 tiers reverse-engineered from Pilkwang's CSV) ----
_YES_SCORE = {1: 0.68, 2: 0.82, 3: 0.94}
def to_scores(verdicts):
    """[(verdict, severity), ...] -> (score, conf). UNK=0.28, NO=0.08."""
    sc, cf = [], []
    for v, s in verdicts:
        if v == "YES":
            sc.append(_YES_SCORE.get(s, 0.82)); cf.append(0.95)
        elif v == "NO":
            sc.append(0.08); cf.append(0.85)
        else:
            sc.append(0.28); cf.append(0.05)
    return sc, cf


class ReportLabeler:
    """Anthropic-compatible endpoint extractor (MiniMax / DeepSeek)."""

    def __init__(self, model="deepseek-v4-flash", base_url=None,
                 cache_path=None, max_tokens=2000):
        import anthropic
        self.client = anthropic.Anthropic(
            api_key=os.environ["ANTHROPIC_API_KEY"],
            base_url=base_url,                 # e.g. https://api.deepseek.com/anthropic
        )
        self.model = model
        self.max_tokens = max_tokens
        self.cache_path = Path(cache_path) if cache_path else None
        self.cache = {}
        self._lock = threading.Lock()

    def _params(self, report):
        return dict(
            model=self.model,
            max_tokens=self.max_tokens,
            thinking={"type": "disabled"},     # non-thinking: ~25x faster, reads bilateral reports too
            system=[{"type": "text", "text": SYSTEM,
                     "cache_control": {"type": "ephemeral"}}],
            messages=[{"role": "user", "content": report.strip()[:12000]}],
        )

    def label(self, uid, report):
        if uid in self.cache:
            return self.cache[uid]
        resp = self.client.messages.create(**self._params(report))
        text = next(b.text for b in resp.content if b.type == "text")
        obj = parse_obj(text)                  # tolerant parse: {key: {"verdict","severity"}}
        verdicts = [(obj[k]["verdict"], obj[k]["severity"] if obj[k]["verdict"] == "YES" else 0)
                    for k in KEYS]
        sc, cf = to_scores(verdicts)
        rec = {"uid": uid, "score": sc, "conf": cf,
               "verdict": [v for v, _ in verdicts],
               "severity": [s for _, s in verdicts]}
        self.cache[uid] = rec
        if self.cache_path:
            with open(self.cache_path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        return rec

    def label_many(self, items, workers=8, progress=None):
        from concurrent.futures import ThreadPoolExecutor
        done = [0]
        def one(it):
            r = self.label(*it)
            done[0] += 1
            if progress and done[0] % progress == 0:
                print(f"  {done[0]}/{len(items)}", flush=True)
            return r
        with ThreadPoolExecutor(workers) as ex:
            return list(ex.map(one, items))


# ---- Usage: full 4,407 studies, DeepSeek flash non-thinking ----
# labeler = ReportLabeler(model="deepseek-v4-flash",
#                         base_url="https://api.deepseek.com/anthropic",
#                         cache_path="labels.jsonl")
# items = [(uid, report_text) for uid, report_text in train[["StudyInstanceUID","Report"]]]
# recs = labeler.label_many(items, workers=32)

## 4. Next steps: disagreement ensemble + conf into training

**No single extractor wins on every label** — each excels somewhere (e.g. one wins ACL/Fracture,
another Effusion/Contusion, another Medial OA, another MCL/Synovitis). **Disagreement carries information.**

### 4.1 Disagreement arbitration
Have multiple models read the same report independently:
- **Consensus** (verdicts agree) → high confidence, take the median score
- **Disagreement** (verdicts differ) → low confidence, lower the conf, or send to an arbiter

This is more likely to improve the score than picking a single "strongest" model, because it
combines each model's strengths instead of betting on one.

### 4.2 conf into training
The extractor already outputs `__conf` (Pilkwang's UNK rows have only 0.05).
Feed it into the loss: **high weight on consensus, low weight on disagreement** — telling the
vision model "these labels are trustworthy, those are not." This is how extraction quality is
actually *transferred* into training.

### 4.3 Validation (the only credible way)
Not just extraction AUC on 58 studies, but **end-to-end**:

Train the same vision model on two weak-label sets and compare final macro AUC:
- **Baseline weak labels**: what we already use — scores from a single model (e.g. DeepSeek flash)
  reading all reports. Think of it as "the status quo, no improvements."
- **Disagreement-ensemble weak labels**: scores after §4.1's multi-model arbitration —
  multiple models read the same report; consensus is trusted, disagreement is down-weighted.
  Think of it as "the improved version."

> Controlled variable: the vision model, data, and training settings are identical;
> **the only difference is the weak-label source**.
> If the ensemble version scores higher → the improvement is real; if it is about the same,
> the ensemble gain was attenuated away during training.

> **Status: this end-to-end validation is not finished yet — experiments are running,
> results expected tomorrow (2026-08-11).**
> Until then, all of the above are **hypotheses** — but hypotheses with direction and a clear
> validation plan are exactly what make them worth proposing.

## 5. Summary

1. **Whether extraction improves the score must be validated end-to-end**, not by the 58-study extraction AUC alone.
2. The 58-study comparison shows: four LLMs cover 58/58, AUC 0.870-0.878.
3. **The real opportunity is disagreement ensemble + conf into training**, not picking one "strongest model".
4. **Next step: run the full report set, and build conf.**
   - **Run the full report set**: apply the extractor (e.g. DeepSeek flash non-thinking) to all 4,407 reports,
     producing a score for every label of every study — not just the 58 gold ones.
   - **Build conf**: each label of each report carries a confidence (`__conf`) saying "how much to trust
     this judgement" — low where models disagree, high where they agree. When training the vision model,
     weight labels by conf so the model trusts high-confidence labels and is not led astray by low-confidence ones.

In [ ]:
print('Done.')